# Sistem Peringatan Dini Banjir Dataran Rendah Berbasis Komputasi Kinerja Tinggi (HPC)
### Studi Kasus dan Implementasi pada Google Colab

**Mata kuliah:** Kuliah Praktisi Informatika
**Rujukan materi:** *"Bridging Global Infrastructure to Address Local Challenge with HPC"* — Asst. Prof. Dr. Worawan Diaz Carballo (Thammasat University), Prodi S1 Teknik Informatika, Universitas Bengkulu.

---

**Cara menjalankan:** pilih menu **Runtime → Run all**. Seluruh notebook berjalan otomatis dari atas ke bawah (±5–8 menit pada runtime CPU gratis). Runtime GPU bersifat opsional: bila tersedia, kernel GPU (CuPy) ikut diuji pada bagian benchmark.

**Daftar isi**
1. Latar belakang dan rumusan studi kasus
2. Landasan teori ringkas
3. Implementasi (tahap Integrator: data → model → simulasi → analisis → laporan)
4. Verifikasi model
5. Benchmark HPC dan skalabilitas
6. Ensemble paralel dan peta probabilitas
7. Buletin peringatan dini
8. Skrip klaster (Slurm)
9. Pembahasan, keterbatasan, dan kesimpulan

## 1. Latar Belakang dan Rumusan Studi Kasus

### 1.1 Keterkaitan dengan materi kuliah
Kuliah praktisi menekankan metafora **"Jembatan"**: menghubungkan infrastruktur superkomputer global dengan persoalan nyata di tingkat lokal. Salah satu contoh yang dibahas adalah **peringatan dini banjir dan bencana hidrometeorologi**, di mana simulasi hidrologi pada superkomputer mempercepat prakiraan dari hitungan hari menjadi hitungan jam, sehingga warga memiliki waktu evakuasi yang cukup. Materi yang sama juga menyebut tiga peran mahasiswa informatika: *Learner*, *Practitioner*, dan *Integrator*.

Studi kasus ini menerjemahkan contoh tersebut menjadi sebuah **prototipe kerja** yang dapat dijalankan penuh di Google Colab.

### 1.2 Masalah
Wilayah dataran rendah dekat pesisir (seperti banyak daerah aliran sungai di Pulau Sumatera) menghadapi dua sumber banjir sekaligus: **luapan sungai** dan **genangan akibat hujan lokal**. Pengambil keputusan perlu menjawab pertanyaan berikut *sebelum* hujan puncak tiba:

1. Area mana yang akan tergenang, seberapa dalam, dan **berapa menit** setelah hujan dimulai?
2. **Berapa penduduk** yang terpapar di tiap zona?
3. Karena prakiraan hujan dan debit sungai selalu mengandung ketidakpastian, **seberapa besar peluang** suatu zona tergenang?

Pertanyaan ke-3 memerlukan banyak simulasi (*ensemble*), dan simulasi 2D beresolusi tinggi mahal secara komputasi. Di sinilah HPC berperan.

### 1.3 Tujuan
- Membangun model genangan banjir 2D (formulasi *local-inertial*) dan memverifikasinya (kekekalan massa, uji keseimbangan).
- Mempercepat komputasi dengan tiga tingkat paralelisme yang dibahas di kuliah: CPU multi-thread (analog OpenMP), GPU (CUDA), dan multi-proses untuk ensemble (analog tugas paralel pada klaster Slurm).
- Menghasilkan produk yang dapat dipakai pemangku kepentingan: peta bahaya, peta probabilitas, estimasi penduduk terpapar, waktu tiba banjir, dan buletin peringatan dini.

### 1.4 Pemetaan terhadap tiga peran mahasiswa informatika

| Peran (dari kuliah) | Kegiatan pada notebook ini | Bagian |
|---|---|---|
| **Learner** | Arsitektur CPU/GPU, kernel numerik, paralelisme *thread* (Numba `prange`), CUDA via CuPy, penjadwalan klaster (Slurm) | 3.2, 5, 8 |
| **Practitioner** | Benchmark kinerja (juta update-sel/detik), analisis *speedup* dan efisiensi, uji skalabilitas | 5, 6 |
| **Integrator** | Merancang alur kerja menyeluruh: data lapangan → konversi format → simulasi masif → analisis → antarmuka laporan untuk pengguna awam | 3.1, 3.7, 7 |

### 1.5 Asumsi dan ruang lingkup (dinyatakan terbuka)
- **Data bersifat sintetis.** DEM, tutupan lahan, dan sebaran penduduk dibangkitkan secara matematis untuk dataran banjir pesisir fiktif seluas 6 × 6 km. Zona A1–D4 adalah zona fiktif, bukan wilayah administratif nyata. Struktur kode sengaja dibuat agar DEM, kekasaran, dan penduduk dari data nyata (mis. DEM nasional dan data kependudukan) dapat langsung menggantikan array sintetis ini.
- Hujan dimodelkan seragam secara spasial dengan koefisien limpasan konstan (0,6) sebagai penyederhanaan infiltrasi dan drainase kota.
- Tepi selatan diperlakukan sebagai outlet (laut/muara) dengan muka air bebas keluar; tepi lain tertutup.
- Hasil studi ini bersifat **demonstrasi metodologi**, bukan prakiraan operasional untuk wilayah tertentu.

## 2. Landasan Teori Ringkas

### 2.1 Model genangan 2D *local-inertial*
Model ini menyederhanakan persamaan air dangkal (*shallow water equations*) dengan mempertahankan suku inersia lokal tetapi mengabaikan suku adveksi, sehingga stabil, ringan, dan banyak dipakai untuk pemodelan genangan (Bates dkk., 2010). Debit per satuan lebar $q$ pada muka sel dihitung dari gradien muka air dan gesekan Manning:

$$
q^{t+\Delta t}_{i+\frac12}=\frac{q^{t}_{i+\frac12}-g\,h_f\,\Delta t\,\dfrac{\partial (h+z)}{\partial x}}{1+\dfrac{g\,\Delta t\,n^{2}\,|q^{t}_{i+\frac12}|}{h_f^{7/3}}}
$$

dengan $h$ kedalaman air, $z$ elevasi dasar, $n$ koefisien Manning, dan $h_f=\max(h_i+z_i,\;h_{i+1}+z_{i+1})-\max(z_i,z_{i+1})$ kedalaman aliran di muka sel. Kekekalan massa pada tiap sel:

$$
h_{i,j}^{t+\Delta t}=h_{i,j}^{t}+\frac{\Delta t}{\Delta x}\left(q_{i-\frac12,j}-q_{i+\frac12,j}+q_{i,j-\frac12}-q_{i,j+\frac12}\right)+(R_{\text{eff}}+S_{\text{sungai}})\,\Delta t
$$

dengan $R_{\text{eff}}$ hujan efektif dan $S_{\text{sungai}}$ debit masuk sungai. Langkah waktu dibatasi kondisi Courant–Friedrichs–Lewy (CFL):

$$
\Delta t\le \alpha\,\frac{\Delta x}{\sqrt{g\,h_{\max}}},\qquad \alpha=0{,}6
$$

Pada kode, debit keluar tiap sel dibatasi agar tidak melebihi volume air yang tersedia, sehingga kedalaman tidak pernah negatif dan massa tetap terjaga.

### 2.2 Mengapa HPC diperlukan?
- Jumlah sel $N=n^2$. Memperhalus grid dua kali lipat menambah sel 4× **dan** memperkecil $\Delta t$ (CFL) sehingga jumlah langkah naik ±2×. Total kerja tumbuh ∝ $n^3$, yaitu ±8× per penghalusan resolusi.
- Prakiraan probabilistik membutuhkan puluhan hingga ratusan simulasi (*ensemble*) yang saling bebas.
- Waktu peringatan sangat singkat: hasil yang datang setelah banjir tiba tidak berguna.

### 2.3 Tiga tingkat paralelisme yang dipakai
| Tingkat | Teknik | Analogi di dunia HPC |
|---|---|---|
| Paralelisme data (stensil per sel) | Numba `prange` (multi-thread) | OpenMP |
| Paralelisme data pada akselerator | CuPy (kernel CUDA) | GPU Nvidia di klaster |
| Paralelisme tugas (ensemble) | Multi-proses (`ProcessPoolExecutor`) | *Job array* / MPI task pada Slurm |

### 2.4 Metrik kinerja
$$
S_p=\frac{T_1}{T_p},\qquad E_p=\frac{S_p}{p},\qquad S(p)=\frac{1}{f+\frac{1-f}{p}}\;\;(\text{hukum Amdahl})
$$
$T_1$ waktu 1 unit komputasi, $T_p$ waktu dengan $p$ unit, $E_p$ efisiensi paralel, dan $f$ fraksi kode yang tetap serial. Throughput kernel dilaporkan dalam **juta update-sel per detik (Mcell/s)**.

## 3. Implementasi

### 3.1 Alur kerja (peran Integrator)

```
Data lapangan (DEM, tutupan lahan, penduduk)  ──►  konversi format (CSV → array 2D)
        │
        ▼
Forcing: hujan + debit sungai  ──►  Simulasi 2D (CPU thread / GPU)  ──►  Ensemble paralel (N skenario)
                                                                              │
                                                                              ▼
              Peta bahaya, peta probabilitas, waktu tiba, penduduk terpapar  ──►  Buletin & laporan HTML
```

### 3.2 Persiapan lingkungan
Sel berikut memeriksa perangkat keras (jumlah core CPU dan GPU) dan memastikan pustaka yang dibutuhkan tersedia.

In [ ]:
import os, sys, platform, subprocess, shutil, time, warnings
warnings.filterwarnings("ignore")
try:
    import numba
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numba"])
    import numba
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML

N_CORES = os.cpu_count()
TRAP = getattr(np, "trapezoid", None) or np.trapz   # kompatibel NumPy 1.x dan 2.x
print("Python        :", platform.python_version())
print("NumPy / SciPy :", np.__version__, "/", scipy.__version__)
print("Numba         :", numba.__version__, "| thread maksimum:", numba.config.NUMBA_NUM_THREADS)
print("Core CPU      :", N_CORES)
if shutil.which("nvidia-smi"):
    print("GPU           :", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                             capture_output=True, text=True).stdout.strip())
else:
    print("GPU           : tidak terdeteksi (notebook tetap berjalan penuh pada CPU)")

os.makedirs("hasil", exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": False, "font.size": 10})
def savefig(fig, name):
    fig.savefig(f"hasil/{name}.png", dpi=110, bbox_inches="tight")

### 3.3 Modul model (`floodsim.py`)
Seluruh model ditulis ke satu berkas modul agar dapat dipakai ulang oleh proses paralel dan skrip klaster. Isi utama modul:

- `make_domain` — membangun DEM sintetis, kekasaran Manning, penduduk, dan zona.
- `make_forcing` — hidrograf hujan dan debit sungai.
- `step_vec` — satu langkah waktu dalam bentuk vektorisasi (NumPy untuk CPU, CuPy untuk GPU; kode yang sama).
- `_step_loops` — kernel yang sama dalam bentuk loop, dikompilasi JIT oleh Numba menjadi versi *serial* dan versi *multi-thread* (`prange`).
- `simulate` — pengendali waktu adaptif (CFL), sumber hujan/sungai, outlet, pelacakan kedalaman maksimum, waktu tiba banjir, dan neraca massa.
- `sample_scenarios`, `run_ensemble` — *Latin Hypercube Sampling* dan eksekusi ensemble paralel multi-proses.
- `lake_at_rest_test`, `benchmark_kernel` — verifikasi dan pengukuran kinerja.

In [ ]:
%%writefile floodsim.py
"""
floodsim.py
Model genangan banjir 2D (formulasi local-inertial / Bates dkk. 2010) untuk studi kasus
"Sistem Peringatan Dini Banjir Dataran Rendah Berbasis HPC".

Backend yang tersedia:
  - numpy          : vektorisasi array (1 core, baseline)
  - numba_serial   : loop terkompilasi JIT, 1 thread
  - numba_parallel : loop terkompilasi JIT, multi-thread (analog OpenMP, prange)
  - cupy           : GPU (CUDA) -- hanya jika CuPy & GPU tersedia
"""
import math
import time
import numpy as np
from scipy.ndimage import gaussian_filter, zoom
import numba as nb
from numba import njit, prange

G = 9.81          # percepatan gravitasi (m/s^2)
HMIN = 1e-3       # kedalaman minimum agar sel dianggap basah (m)
DT_MAX = 8.0      # batas atas langkah waktu (s)
DT_MIN = 0.25     # batas bawah langkah waktu (s)
CFL = 0.6         # bilangan Courant

# ----------------------------------------------------------------------------------
# 1. DOMAIN (DEM sintetis, tutupan lahan, penduduk, zona)
# ----------------------------------------------------------------------------------
def make_domain(n=240, L=6000.0, seed=7):
    """Membangun domain dataran banjir sintetis berukuran L x L meter dengan n x n sel.
    Baris 0 = utara (hulu, tinggi); baris terakhir = selatan (pesisir, rendah)."""
    dx = L / n
    c = (np.arange(n) + 0.5) / n
    Y, X = np.meshgrid(c, c, indexing="ij")

    base = 14.0 * (1.0 - Y) ** 2.2 + 0.15                     # lereng dataran: 14 m -> ~0 m
    xc = 0.5 + 0.16 * np.sin(2 * np.pi * 1.4 * Y + 0.6)        # sumbu sungai meander
    dist = np.abs(X - xc) * L                                  # jarak lateral ke sumbu sungai (m)
    W = 120.0                                                  # lebar sungai (m)
    chan = np.exp(-(dist / (W / 2.0)) ** 4)
    levee = 0.6 * np.exp(-((dist - (W / 2.0 + 40.0)) / 40.0) ** 2)

    rng = np.random.default_rng(seed)
    noise = gaussian_filter(rng.normal(size=(64, 64)), 3)
    noise = zoom(noise, n / 64.0, order=3)[:n, :n]
    noise = noise / noise.std() * 0.35
    patch = gaussian_filter(rng.normal(size=(64, 64)), 4)
    patch = zoom(patch, n / 64.0, order=3)[:n, :n]
    patch = patch / patch.std()

    z = base + noise + levee - 3.5 * chan

    chan_mask = dist < 0.45 * W
    urban = (dist < 1800.0) & (Y > 0.35) & (patch > -0.6) & (~chan_mask)
    nman = np.full((n, n), 0.08)           # lahan pertanian / vegetasi
    nman[urban] = 0.06                     # permukiman
    nman[chan_mask] = 0.035                # alur sungai

    density = np.where(urban, 0.007, 0.0003)   # jiwa / m^2 (70 jiwa/ha di permukiman)
    density[chan_mask] = 0.0
    pop = density * dx * dx                    # jiwa per sel

    rows = np.arange(n)[:, None] * 4 // n
    cols = np.arange(n)[None, :] * 4 // n
    zone = (rows * 4 + cols).astype(np.int32)
    zone_names = [f"{'ABCD'[r]}{c + 1}" for r in range(4) for c in range(4)]

    k_out = max(1, int(round(150.0 / dx)))     # sel outlet (laut) di tepi selatan
    j_in = 1
    idx = np.where(dist[j_in, :] < 0.45 * W)[0]
    if idx.size == 0:
        idx = np.array([int(np.argmin(dist[j_in, :]))])
    return dict(n=n, L=L, dx=dx, z=z, nman=nman, pop=pop, urban=urban, chan=chan_mask,
                zone=zone, zone_names=zone_names, k_out=k_out,
                inflow=(j_in, int(idx.min()), int(idx.max()) + 1), dist=dist)

# ----------------------------------------------------------------------------------
# 2. FORCING (hujan + debit sungai)
# ----------------------------------------------------------------------------------
def make_forcing(P_total_mm, t_peak_h, q_peak, duration_h=6.0, runoff_coeff=0.6,
                 q_base=40.0, lag_h=0.75, sigma_h=0.9):
    """Hyetograph Gaussian (total P_total_mm) dan hidrograf debit masuk sungai (m^3/s)."""
    t = np.arange(0.0, duration_h * 3600.0 + 1.0, 60.0)
    shape = np.exp(-0.5 * ((t / 3600.0 - t_peak_h) / sigma_h) ** 2)
    shape /= shape.sum() * 60.0
    rain = runoff_coeff * (P_total_mm / 1000.0) * shape                # m/s (hujan efektif)
    q = q_base + (q_peak - q_base) * np.exp(
        -0.5 * ((t / 3600.0 - (t_peak_h + lag_h)) / (1.2 * sigma_h)) ** 2)
    return dict(t=t, rain=rain, q=q, P_total_mm=P_total_mm, t_peak_h=t_peak_h, q_peak=q_peak)

# ----------------------------------------------------------------------------------
# 3. KERNEL NUMERIK
# ----------------------------------------------------------------------------------
def step_vec(xp, h, z, nman, qx, qy, dt, dx):
    """Satu langkah waktu, versi vektorisasi (NumPy / CuPy)."""
    H = z + h
    # --- fluks arah x (muka sel antara kolom i dan i+1)
    hf = xp.maximum(H[:, :-1], H[:, 1:]) - xp.maximum(z[:, :-1], z[:, 1:])
    wet = hf > HMIN
    hfs = xp.where(wet, hf, 1.0)
    nf = 0.5 * (nman[:, :-1] + nman[:, 1:])
    q = (qx - G * hfs * dt * (H[:, 1:] - H[:, :-1]) / dx) / (
        1.0 + G * dt * nf * nf * xp.abs(qx) / hfs ** (7.0 / 3.0))
    qlim = xp.where(q > 0, h[:, :-1], h[:, 1:]) * dx / (4.0 * dt)
    qx[...] = xp.where(wet, xp.clip(q, -qlim, qlim), 0.0)
    # --- fluks arah y
    hf = xp.maximum(H[:-1, :], H[1:, :]) - xp.maximum(z[:-1, :], z[1:, :])
    wet = hf > HMIN
    hfs = xp.where(wet, hf, 1.0)
    nf = 0.5 * (nman[:-1, :] + nman[1:, :])
    q = (qy - G * hfs * dt * (H[1:, :] - H[:-1, :]) / dx) / (
        1.0 + G * dt * nf * nf * xp.abs(qy) / hfs ** (7.0 / 3.0))
    qlim = xp.where(q > 0, h[:-1, :], h[1:, :]) * dx / (4.0 * dt)
    qy[...] = xp.where(wet, xp.clip(q, -qlim, qlim), 0.0)
    # --- kontinuitas massa
    dh = xp.zeros_like(h)
    dh[:, :-1] -= qx
    dh[:, 1:] += qx
    dh[:-1, :] -= qy
    dh[1:, :] += qy
    h += (dt / dx) * dh
    xp.maximum(h, 0.0, out=h)


def _step_loops(h, z, nman, qx, qy, dt, dx):
    """Kernel yang sama dalam bentuk loop eksplisit (dikompilasi Numba)."""
    ny, nx = h.shape
    for j in prange(ny):
        for i in range(nx - 1):
            h0 = h[j, i]
            h1 = h[j, i + 1]
            z0 = z[j, i]
            z1 = z[j, i + 1]
            H0 = z0 + h0
            H1 = z1 + h1
            hf = max(H0, H1) - max(z0, z1)
            if hf > HMIN:
                nf = 0.5 * (nman[j, i] + nman[j, i + 1])
                q0 = qx[j, i]
                q = (q0 - G * hf * dt * (H1 - H0) / dx) / (
                    1.0 + G * dt * nf * nf * abs(q0) / hf ** (7.0 / 3.0))
                if q > 0.0:
                    ql = h0 * dx / (4.0 * dt)
                    if q > ql:
                        q = ql
                else:
                    ql = h1 * dx / (4.0 * dt)
                    if q < -ql:
                        q = -ql
                qx[j, i] = q
            else:
                qx[j, i] = 0.0
    for j in prange(ny - 1):
        for i in range(nx):
            h0 = h[j, i]
            h1 = h[j + 1, i]
            z0 = z[j, i]
            z1 = z[j + 1, i]
            H0 = z0 + h0
            H1 = z1 + h1
            hf = max(H0, H1) - max(z0, z1)
            if hf > HMIN:
                nf = 0.5 * (nman[j, i] + nman[j + 1, i])
                q0 = qy[j, i]
                q = (q0 - G * hf * dt * (H1 - H0) / dx) / (
                    1.0 + G * dt * nf * nf * abs(q0) / hf ** (7.0 / 3.0))
                if q > 0.0:
                    ql = h0 * dx / (4.0 * dt)
                    if q > ql:
                        q = ql
                else:
                    ql = h1 * dx / (4.0 * dt)
                    if q < -ql:
                        q = -ql
                qy[j, i] = q
            else:
                qy[j, i] = 0.0
    for j in prange(ny):
        for i in range(nx):
            dh = 0.0
            if i > 0:
                dh += qx[j, i - 1]
            if i < nx - 1:
                dh -= qx[j, i]
            if j > 0:
                dh += qy[j - 1, i]
            if j < ny - 1:
                dh -= qy[j, i]
            v = h[j, i] + dt / dx * dh
            h[j, i] = v if v > 0.0 else 0.0


step_nb_serial = njit(fastmath=True)(_step_loops)
step_nb_parallel = njit(parallel=True, fastmath=True)(_step_loops)


def get_backend(name):
    """Mengembalikan (xp, fungsi_step, fungsi_sinkron)."""
    if name == "numpy":
        return np, (lambda h, z, nm, qx, qy, dt, dx: step_vec(np, h, z, nm, qx, qy, dt, dx)), (lambda: None)
    if name == "numba_serial":
        return np, step_nb_serial, (lambda: None)
    if name == "numba_parallel":
        return np, step_nb_parallel, (lambda: None)
    if name == "cupy":
        import cupy as cp
        return cp, (lambda h, z, nm, qx, qy, dt, dx: step_vec(cp, h, z, nm, qx, qy, dt, dx)), \
            (lambda: cp.cuda.Stream.null.synchronize())
    raise ValueError(f"backend tidak dikenal: {name}")


def available_backends():
    out = ["numpy", "numba_serial", "numba_parallel"]
    try:
        import cupy as cp
        if cp.cuda.runtime.getDeviceCount() > 0:
            out.append("cupy")
    except Exception:
        pass
    return out

# ----------------------------------------------------------------------------------
# 4. SIMULASI
# ----------------------------------------------------------------------------------
def simulate(dom, forcing, backend="numba_parallel", duration_h=6.0, snap_every_min=None,
             hthr=0.3):
    """Menjalankan simulasi banjir. Mengembalikan peta kedalaman maksimum, waktu tiba banjir,
    neraca massa, dan (opsional) cuplikan kedalaman tiap snap_every_min menit."""
    xp, step, sync = get_backend(backend)
    dx = dom["dx"]
    z = xp.asarray(dom["z"])
    nman = xp.asarray(dom["nman"])
    ny, nx = z.shape
    h = xp.zeros((ny, nx))
    qx = xp.zeros((ny, nx - 1))
    qy = xp.zeros((ny - 1, nx))
    hmax = xp.zeros((ny, nx))
    toa = xp.full((ny, nx), -1.0)
    j_in, i0, i1 = dom["inflow"]
    n_in = i1 - i0
    k = dom["k_out"]
    area = dx * dx
    T = duration_h * 3600.0
    t, it, dt = 0.0, 0, 1.0
    vol_in = vol_out = 0.0
    snaps, snap_t = [], []
    next_snap = 0.0
    t0 = time.perf_counter()
    while t < T - 1e-9:
        if it % 10 == 0:
            hm = float(h.max())
            dt = min(DT_MAX, max(DT_MIN, CFL * dx / math.sqrt(G * max(hm, 0.1))))
        dt = min(dt, T - t)
        r = float(np.interp(t, forcing["t"], forcing["rain"]))
        qin = float(np.interp(t, forcing["t"], forcing["q"]))
        h += r * dt
        h[j_in, i0:i1] += qin * dt / (n_in * area)
        vol_in += (r * area * ny * nx + qin) * dt
        step(h, z, nman, qx, qy, dt, dx)
        out = h[ny - k:, :]
        vol_out += float(out.sum()) * area
        out[...] = 0.0
        t += dt
        it += 1
        if it % 10 == 0:
            hmax = xp.maximum(hmax, h)
            toa = xp.where((toa < 0) & (h > hthr), t, toa)
        if snap_every_min is not None and t >= next_snap:
            snaps.append(np.asarray(h.get() if backend == "cupy" else h, dtype=np.float32).copy())
            snap_t.append(t / 3600.0)
            next_snap += snap_every_min * 60.0
    sync()
    runtime = time.perf_counter() - t0
    conv = (lambda a: a.get()) if backend == "cupy" else (lambda a: a)
    h_end = conv(h)
    vol_end = float(h_end.sum()) * area
    mass_err = abs(vol_in - vol_out - vol_end) / max(vol_in, 1.0)
    toa_np = conv(toa).astype(np.float64)
    toa_np[toa_np < 0] = np.nan
    return dict(hmax=conv(hmax), toa=toa_np, h_end=h_end, steps=it, runtime=runtime,
                vol_in=vol_in, vol_out=vol_out, vol_end=vol_end, mass_err=mass_err,
                snaps=snaps, snap_t=snap_t, backend=backend)

# ----------------------------------------------------------------------------------
# 5. ENSEMBLE (paralel level skenario / "embarrassingly parallel")
# ----------------------------------------------------------------------------------
def sample_scenarios(n_scen, seed=2024, P_range=(90, 250), Q_range=(250, 900), tp_range=(1.5, 3.5)):
    """Latin Hypercube Sampling untuk curah hujan total, debit puncak sungai, dan waktu puncak hujan."""
    rng = np.random.default_rng(seed)
    lhs = lambda: (rng.permutation(n_scen) + rng.random(n_scen)) / n_scen
    P = P_range[0] + lhs() * (P_range[1] - P_range[0])
    Q = Q_range[0] + lhs() * (Q_range[1] - Q_range[0])
    tp = tp_range[0] + lhs() * (tp_range[1] - tp_range[0])
    return [dict(id=i, P=float(P[i]), Q=float(Q[i]), tp=float(tp[i])) for i in range(n_scen)]


def run_scenario(args):
    """Fungsi kerja untuk satu skenario (dapat dikirim ke proses lain)."""
    sc, n, duration_h, backend = args
    dom = make_domain(n)
    fc = make_forcing(sc["P"], sc["tp"], sc["Q"], duration_h=duration_h)
    res = simulate(dom, fc, backend=backend, duration_h=duration_h)
    return dict(id=sc["id"], P=sc["P"], Q=sc["Q"], tp=sc["tp"],
                hmax=res["hmax"].astype(np.float32), toa=res["toa"].astype(np.float32),
                runtime=res["runtime"], mass_err=res["mass_err"], steps=res["steps"])


def run_ensemble(scenarios, n=240, duration_h=6.0, workers=1, backend="numba_serial"):
    """Menjalankan seluruh skenario secara serial (workers=1) atau paralel multi-proses."""
    jobs = [(sc, n, duration_h, backend) for sc in scenarios]
    t0 = time.perf_counter()
    if workers <= 1:
        results = [run_scenario(j) for j in jobs]
    else:
        import multiprocessing as mp
        from concurrent.futures import ProcessPoolExecutor
        with ProcessPoolExecutor(max_workers=workers, mp_context=mp.get_context("spawn")) as ex:
            results = list(ex.map(run_scenario, jobs))
    return results, time.perf_counter() - t0

# ----------------------------------------------------------------------------------
# 6. VERIFIKASI & BENCHMARK
# ----------------------------------------------------------------------------------
def lake_at_rest_test(n=120, backend="numba_serial", steps=300):
    """Uji 'danau diam': permukaan air datar di atas dasar tak rata harus tetap diam (well-balanced)."""
    dom = make_domain(n)
    xp, step, sync = get_backend(backend)
    z = xp.asarray(dom["z"])
    nman = xp.asarray(dom["nman"])
    eta = 3.0                                                    # elevasi muka air datar (m)
    h = xp.maximum(eta - z, 0.0)
    qx = xp.zeros((n, n - 1))
    qy = xp.zeros((n - 1, n))
    dt = 0.5
    for _ in range(steps):
        step(h, z, nman, qx, qy, dt, dom["dx"])
    qmax = float(max(abs(qx).max(), abs(qy).max()))
    return qmax


def benchmark_kernel(n, steps, backend, dt=1.0):
    """Mengukur waktu 'steps' langkah kernel pada grid n x n (tanpa I/O)."""
    dom = make_domain(n)
    xp, step, sync = get_backend(backend)
    z = xp.asarray(dom["z"])
    nman = xp.asarray(dom["nman"])
    h = xp.maximum(1.0 - z, 0.0) + 0.05
    qx = xp.zeros((n, n - 1))
    qy = xp.zeros((n - 1, n))
    step(h, z, nman, qx, qy, dt, dom["dx"])       # pemanasan (kompilasi JIT / alokasi GPU)
    sync()
    t0 = time.perf_counter()
    for _ in range(steps):
        step(h, z, nman, qx, qy, dt, dom["dx"])
    sync()
    el = time.perf_counter() - t0
    return el, n * n * steps / el / 1e6          # detik, juta update-sel per detik

In [ ]:
import importlib
importlib.invalidate_caches()
import floodsim as fs
importlib.reload(fs)

BACKENDS = fs.available_backends()
print("Backend yang tersedia:", BACKENDS)

# Pemanasan kompilasi JIT (agar waktu kompilasi tidak tercampur dalam pengukuran)
_dom = fs.make_domain(60); _fc = fs.make_forcing(100, 1.0, 100, duration_h=0.05)
for b in ("numba_serial", "numba_parallel"):
    fs.simulate(_dom, _fc, backend=b, duration_h=0.05)
print("Kompilasi JIT selesai.")

### 3.4 Tahap data: domain studi dan konversi format
Dataran banjir fiktif berukuran 6 × 6 km (grid 240 × 240, resolusi 25 m). Sungai meander mengalir dari utara (hulu, elevasi tinggi) ke selatan (pesisir, hampir datar). Permukiman berkumpul di sekitar sungai di bagian tengah–selatan.

Sesuai peran *Integrator*, data terlebih dahulu **diekspor sebagai CSV mentah** (format yang lazim diterima dari lapangan atau instansi), kemudian **dikonversi kembali menjadi array 2D** (format yang dibutuhkan simulator), disertai pemeriksaan kualitas data.

In [ ]:
N = 240
dom0 = fs.make_domain(N)

# --- 1) Ekspor ke format "mentah" (tabel panjang CSV)
os.makedirs("data_lapangan", exist_ok=True)
jj, ii = np.meshgrid(np.arange(N), np.arange(N), indexing="ij")
raw = pd.DataFrame({
    "baris": jj.ravel(), "kolom": ii.ravel(),
    "elevasi_m": dom0["z"].ravel().round(3),
    "manning_n": dom0["nman"].ravel().round(3),
    "penduduk_jiwa": dom0["pop"].ravel().round(4),
    "zona": [dom0["zone_names"][k] for k in dom0["zone"].ravel()],
})
raw.to_csv("data_lapangan/dem_penduduk_mentah.csv", index=False)
print(f"CSV mentah: {len(raw):,} baris, {os.path.getsize('data_lapangan/dem_penduduk_mentah.csv')/1e6:.1f} MB")

# --- 2) Baca kembali dan konversi ke array 2D untuk simulator
df = pd.read_csv("data_lapangan/dem_penduduk_mentah.csv")
to2d = lambda col: df.pivot(index="baris", columns="kolom", values=col).to_numpy()
z_arr, n_arr, p_arr = to2d("elevasi_m"), to2d("manning_n"), to2d("penduduk_jiwa")

# --- 3) Pemeriksaan kualitas data
print("Nilai kosong (NaN)         :", int(np.isnan(z_arr).sum() + np.isnan(n_arr).sum() + np.isnan(p_arr).sum()))
print("Rentang elevasi (m)        : %.2f s.d. %.2f" % (z_arr.min(), z_arr.max()))
print("Rentang Manning n          : %.3f s.d. %.3f" % (n_arr.min(), n_arr.max()))
print("Total penduduk domain      : %s jiwa" % format(int(p_arr.sum()), ","))
print("Selisih maks. akibat pembulatan CSV (elevasi): %.4f m" % np.abs(z_arr - dom0["z"]).max())

# Domain yang dipakai simulasi dasar berasal dari data hasil konversi
dom = dict(dom0); dom.update(z=z_arr, nman=n_arr, pop=p_arr)
np.savez_compressed("data_lapangan/domain_simulasi.npz", z=z_arr, nman=n_arr, pop=p_arr, zone=dom["zone"])

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

fig, ax = plt.subplots(2, 2, figsize=(11, 9.5))
L = dom["L"] / 1000.0
ext = [0, L, L, 0]

im = ax[0, 0].imshow(dom["z"], cmap="terrain", extent=ext, vmin=-3, vmax=14)
ax[0, 0].set_title("(a) Elevasi dasar / DEM (m)"); plt.colorbar(im, ax=ax[0, 0], shrink=0.8)

cover = np.where(dom["chan"], 0, np.where(dom["urban"], 1, 2))
ax[0, 1].imshow(cover, cmap=ListedColormap(["#2b7bba", "#d95f02", "#9ccc65"]), extent=ext, interpolation="nearest")
ax[0, 1].set_title("(b) Tutupan lahan / kekasaran Manning")
ax[0, 1].legend(handles=[Patch(color="#2b7bba", label="Sungai (n=0,035)"),
                         Patch(color="#d95f02", label="Permukiman (n=0,06)"),
                         Patch(color="#9ccc65", label="Vegetasi/pertanian (n=0,08)")],
                loc="lower left", fontsize=8)

dens = dom["pop"] / (dom["dx"] ** 2) * 1e4
im = ax[1, 0].imshow(dens, cmap="YlOrRd", extent=ext, vmax=80)
ax[1, 0].set_title("(c) Kepadatan penduduk (jiwa/ha)"); plt.colorbar(im, ax=ax[1, 0], shrink=0.8)

ax[1, 1].imshow(dom["z"], cmap="Greys", extent=ext, alpha=0.5, vmin=-3, vmax=14)
for k in range(1, 4):
    ax[1, 1].axhline(k * L / 4, color="k", lw=0.8); ax[1, 1].axvline(k * L / 4, color="k", lw=0.8)
for r in range(4):
    for c in range(4):
        ax[1, 1].text((c + 0.5) * L / 4, (r + 0.5) * L / 4, dom["zone_names"][r * 4 + c], ha="center", va="center",
                      fontsize=13, fontweight="bold")
ax[1, 1].set_title("(d) Zona analisis (fiktif)")
for a in ax.ravel():
    a.set_xlabel("x (km)"); a.set_ylabel("y (km), utara → selatan")
fig.suptitle("Domain studi kasus: dataran banjir pesisir sintetis 6 × 6 km", fontsize=13)
plt.tight_layout(); savefig(fig, "01_domain"); plt.show()

### 3.5 Forcing: skenario hujan dan debit sungai
**Skenario Dasar:** hujan ekstrem total 180 mm dalam 6 jam (puncak pada jam ke-2,5) dan debit puncak sungai 600 m³/s yang tiba 45 menit setelah puncak hujan. Hujan efektif = 0,6 × hujan total.

In [ ]:
P_BASE, TP_BASE, Q_BASE = 180.0, 2.5, 600.0
fc = fs.make_forcing(P_BASE, TP_BASE, Q_BASE)

fig, ax = plt.subplots(figsize=(9, 3.8))
th = fc["t"] / 3600
ax.fill_between(th, fc["rain"] / 0.6 * 3.6e6, color="#4c78a8", alpha=0.7, label="Intensitas hujan (mm/jam)")
ax.set_xlabel("Waktu sejak awal simulasi (jam)"); ax.set_ylabel("Intensitas hujan (mm/jam)", color="#4c78a8")
ax2 = ax.twinx(); ax2.plot(th, fc["q"], color="#e45756", lw=2.2, label="Debit sungai masuk (m³/s)")
ax2.set_ylabel("Debit sungai (m³/s)", color="#e45756")
ax.set_title("Skenario Dasar: hujan dan debit sungai"); fig.tight_layout(); savefig(fig, "02_forcing"); plt.show()

rain_total_mm = TRAP(fc["rain"], fc["t"]) / 0.6 * 1000
inflow_mm3 = TRAP(fc["q"], fc["t"]) / 1e6
print(f"Hujan total terintegrasi : {rain_total_mm:.1f} mm  (efektif {rain_total_mm*0.6:.1f} mm)")
print(f"Volume debit sungai masuk: {inflow_mm3:.2f} juta m³")

## 4. Verifikasi Model
Sebelum hasil dipakai, model diuji dengan tiga pemeriksaan standar:

1. **Uji danau diam (*lake at rest*)**: permukaan air datar di atas dasar yang tidak rata harus tetap diam. Model yang tidak seimbang menghasilkan arus palsu.
2. **Uji hujan pada cekungan tertutup datar**: kedalaman akhir harus sama dengan hujan efektif yang jatuh.
3. **Neraca massa pada simulasi penuh** (diperiksa di bagian 4.1): *volume masuk − volume keluar − volume tersisa ≈ 0*.

In [ ]:
# --- Uji 1: danau diam
q_resid = fs.lake_at_rest_test(n=120, backend="numba_serial", steps=300)
print(f"Uji 1 | Debit sisa maksimum pada danau diam : {q_resid:.2e} m²/s  ->", "LULUS" if q_resid < 1e-8 else "GAGAL")

# --- Uji 2: hujan di atas cekungan tertutup datar (tanpa outlet, tanpa sungai)
flat = dict(dx=50.0, z=np.zeros((60, 60)), nman=np.full((60, 60), 0.03), k_out=0, inflow=(1, 5, 6))
fc_t = fs.make_forcing(100.0, 2.0, 0.0, q_base=0.0)
r_t = fs.simulate(flat, fc_t, backend="numba_serial")
expected = TRAP(fc_t["rain"], fc_t["t"])
got = r_t["h_end"].mean()
rel = abs(got - expected) / expected
print(f"Uji 2 | Kedalaman akhir: {got*1000:.2f} mm vs hujan efektif: {expected*1000:.2f} mm (selisih relatif {rel:.2%})  ->",
      "LULUS" if rel < 0.01 else "GAGAL")
print(f"        Keseragaman kedalaman (std): {r_t['h_end'].std():.2e} m")

### 4.1 Simulasi Skenario Dasar (kernel multi-thread)
Simulasi 6 jam waktu nyata dijalankan dengan kernel Numba multi-thread. Cuplikan kedalaman disimpan tiap 15 menit untuk visualisasi dan animasi.

In [ ]:
res = fs.simulate(dom, fc, backend="numba_parallel", snap_every_min=15)
nonchan = ~dom["chan"]
print(f"Backend             : {res['backend']}  ({numba.get_num_threads()} thread)")
print(f"Waktu komputasi     : {res['runtime']:.1f} detik untuk 6 jam waktu nyata")
print(f"Jumlah langkah waktu: {res['steps']:,}  (rata-rata Δt = {6*3600/res['steps']:.2f} s)")
print(f"Volume masuk        : {res['vol_in']/1e6:.3f} juta m³")
print(f"Volume keluar       : {res['vol_out']/1e6:.3f} juta m³ (outlet)")
print(f"Volume tersisa      : {res['vol_end']/1e6:.3f} juta m³")
print(f"Galat neraca massa  : {res['mass_err']:.2e}  ->", "LULUS" if res["mass_err"] < 1e-6 else "GAGAL")
print(f"Rasio waktu komputasi / waktu nyata: 1 : {6*3600/res['runtime']:.0f}")

In [ ]:
snaps, st = np.array(res["snaps"]), np.array(res["snap_t"])
targets = [1, 2, 3, 4, 5, 6]
fig, ax = plt.subplots(2, 3, figsize=(13, 8.5))
for a, tt in zip(ax.ravel(), targets):
    k = int(np.argmin(np.abs(st - tt)))
    a.imshow(dom["z"], cmap="Greys", extent=ext, alpha=0.45, vmin=-3, vmax=14)
    h = np.ma.masked_less(snaps[k], 0.05)
    im = a.imshow(h, cmap="Blues", vmin=0, vmax=2.0, extent=ext)
    a.set_title(f"t = {st[k]:.2f} jam"); a.set_xlabel("x (km)"); a.set_ylabel("y (km)")
fig.colorbar(im, ax=ax, shrink=0.6, label="Kedalaman genangan (m)")
fig.suptitle("Evolusi genangan banjir — Skenario Dasar", fontsize=13)
savefig(fig, "03_evolusi"); plt.show()

In [ ]:
from matplotlib import animation
plt.rcParams["animation.embed_limit"] = 64
fig, a = plt.subplots(figsize=(5.4, 4.8))
a.imshow(dom["z"], cmap="Greys", extent=ext, alpha=0.45, vmin=-3, vmax=14)
imw = a.imshow(np.ma.masked_less(snaps[0], 0.05), cmap="Blues", vmin=0, vmax=2.0, extent=ext)
ttl = a.set_title("t = 0,00 jam"); a.set_xlabel("x (km)"); a.set_ylabel("y (km)")
fig.colorbar(imw, ax=a, shrink=0.8, label="Kedalaman (m)")
def _upd(i):
    imw.set_data(np.ma.masked_less(snaps[i], 0.05)); ttl.set_text(f"t = {st[i]:.2f} jam"); return imw, ttl
ani = animation.FuncAnimation(fig, _upd, frames=len(snaps), interval=250)
plt.close(fig)
HTML(ani.to_jshtml())

### 4.2 Peta bahaya, waktu tiba banjir, dan penduduk terpapar
Kedalaman maksimum diklasifikasikan menjadi lima kelas (klasifikasi sederhana untuk studi ini). Alur sungai dikecualikan dari analisis karena terisi air pada kondisi normal. **Penduduk terpapar** didefinisikan sebagai penduduk pada sel dengan kedalaman maksimum > 0,5 m.

In [ ]:
bounds = [0, 0.1, 0.3, 0.5, 1.0, 10]
cls_names = ["Aman (<0,1 m)", "Rendah (0,1–0,3 m)", "Sedang (0,3–0,5 m)", "Tinggi (0,5–1,0 m)", "Sangat tinggi (>1,0 m)"]
cls_cols = ["#f7f7f7", "#c6dbef", "#fdd835", "#fb8c00", "#c62828"]

hm = np.where(dom["chan"], np.nan, res["hmax"])
fig, ax = plt.subplots(1, 3, figsize=(17, 5.4))
ax[0].imshow(dom["z"], cmap="Greys", extent=ext, alpha=0.35, vmin=-3, vmax=14)
ax[0].imshow(np.ma.masked_invalid(hm), cmap=ListedColormap(cls_cols), norm=BoundaryNorm(bounds, 5), extent=ext, interpolation="nearest")
ax[0].contour(np.linspace(0, L, N), np.linspace(0, L, N), dom["urban"].astype(float), levels=[0.5], colors="k", linewidths=0.5)
ax[0].set_title("(a) Kelas bahaya banjir (kedalaman maksimum)")
ax[0].legend(handles=[Patch(color=c, label=n) for c, n in zip(cls_cols, cls_names)], loc="lower left", fontsize=7.5)

toa_min = res["toa"] / 60.0
im = ax[1].imshow(np.ma.masked_invalid(np.where(dom["chan"], np.nan, toa_min)), cmap="viridis_r", extent=ext)
ax[1].set_title("(b) Waktu tiba banjir > 0,3 m (menit)"); plt.colorbar(im, ax=ax[1], shrink=0.8)

im = ax[2].imshow(np.ma.masked_invalid(hm), cmap="Blues", vmin=0, vmax=2.5, extent=ext)
ax[2].set_title("(c) Kedalaman maksimum (m)"); plt.colorbar(im, ax=ax[2], shrink=0.8)
for a in ax: a.set_xlabel("x (km)"); a.set_ylabel("y (km)")
fig.suptitle("Peta bahaya — Skenario Dasar (hujan 180 mm/6 jam, debit puncak 600 m³/s)", fontsize=13)
plt.tight_layout(); savefig(fig, "04_peta_bahaya_dasar"); plt.show()

cell_km2 = (dom["dx"] / 1000) ** 2
tab = []
for i, nm in enumerate(cls_names):
    m = (np.nan_to_num(hm, nan=-1) >= bounds[i]) & (np.nan_to_num(hm, nan=-1) < bounds[i + 1]) & nonchan
    tab.append([nm, m.sum() * cell_km2, dom["pop"][m].sum()])
tab = pd.DataFrame(tab, columns=["Kelas bahaya", "Luas (km²)", "Penduduk (jiwa)"])
tab["Luas (km²)"] = tab["Luas (km²)"].round(2); tab["Penduduk (jiwa)"] = tab["Penduduk (jiwa)"].round(0).astype(int)
display(tab)

In [ ]:
def zone_table(hmax_map, dom, thr=0.5):
    zid = dom["zone"].ravel()
    pop_z = np.bincount(zid, weights=dom["pop"].ravel(), minlength=16)
    exp_z = np.bincount(zid, weights=(dom["pop"] * ((hmax_map > thr) & ~dom["chan"])).ravel(), minlength=16)
    area_z = np.bincount(zid, weights=(((hmax_map > thr) & ~dom["chan"]).ravel() * dom["dx"] ** 2 / 1e6), minlength=16)
    return pop_z, exp_z, area_z

pop_z, exp_z, area_z = zone_table(res["hmax"], dom)
zt = pd.DataFrame({"Zona": dom["zone_names"], "Penduduk zona": pop_z.round(0).astype(int),
                   "Terpapar (>0,5 m)": exp_z.round(0).astype(int),
                   "Persen terpapar": (100 * exp_z / np.maximum(pop_z, 1)).round(1),
                   "Luas tergenang (km²)": area_z.round(3)}).sort_values("Terpapar (>0,5 m)", ascending=False)
display(zt.head(8).reset_index(drop=True))
print(f"Total penduduk terpapar (Skenario Dasar): {int(exp_z.sum()):,} jiwa dari {int(pop_z.sum()):,} jiwa ({100*exp_z.sum()/pop_z.sum():.1f}%)")

## 5. Benchmark HPC dan Skalabilitas (peran Practitioner)
Kernel numerik yang sama diukur pada beberapa implementasi dan ukuran grid. Ukuran kinerja: **juta update-sel per detik (Mcell/s)**; makin besar makin baik. Pengukuran mengecualikan waktu kompilasi JIT.

### 5.1 Perbandingan implementasi dan ukuran grid

In [ ]:
sizes, steps = [128, 256, 512, 1024], 40
rows = []
for b in BACKENDS:
    for n in sizes:
        try:
            el, thr = fs.benchmark_kernel(n, steps, b)
            rows.append(dict(backend=b, grid=n, detik=el, Mcell_s=thr))
        except Exception as e:
            print(f"[lewati] {b} @ {n}: {type(e).__name__}: {e}")
bm = pd.DataFrame(rows)
piv = bm.pivot(index="grid", columns="backend", values="Mcell_s")[[b for b in BACKENDS if b in set(bm.backend)]]
display(piv.round(1).rename_axis("Grid (n×n)").rename(columns={
    "numpy": "NumPy (1 core)", "numba_serial": "Numba serial", "numba_parallel": "Numba multi-thread", "cupy": "GPU (CuPy)"}))

lab = {"numpy": "NumPy (vektorisasi, 1 core)", "numba_serial": "Numba serial (1 thread)",
       "numba_parallel": f"Numba paralel ({numba.config.NUMBA_NUM_THREADS} thread)", "cupy": "GPU (CuPy/CUDA)"}
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
for b in piv.columns:
    ax[0].plot(piv.index, piv[b], "o-", lw=2, label=lab[b])
ax[0].set_xscale("log", base=2); ax[0].set_yscale("log")
ax[0].set_xlabel("Ukuran grid n (n × n sel)"); ax[0].set_ylabel("Throughput (Mcell/s)"); ax[0].legend(fontsize=8)
ax[0].set_title("(a) Throughput kernel"); ax[0].grid(True, which="both", alpha=0.3)
big = piv.loc[piv.index.max()]
sp = (big / big["numpy"]).drop("numpy")
sp_all = pd.concat([pd.Series({"numpy": 1.0}), sp])
bars = ax[1].bar([lab[b].split(" (")[0] for b in sp_all.index], sp_all.values, color=["#9e9e9e", "#4c78a8", "#59a14f", "#e15759"][:len(sp_all)])
for r_, v in zip(bars, sp_all.values): ax[1].text(r_.get_x() + r_.get_width() / 2, v, f"{v:.1f}×", ha="center", va="bottom")
ax[1].set_ylabel("Percepatan relatif terhadap NumPy"); ax[1].set_title(f"(b) Speedup pada grid {piv.index.max()}×{piv.index.max()}")
plt.tight_layout(); savefig(fig, "05_benchmark_backend"); plt.show()

### 5.2 Skalabilitas jumlah thread (*strong scaling*)
Ukuran masalah dibuat tetap (grid 768 × 768), jumlah thread dinaikkan. *Speedup* $S_p=T_1/T_p$, efisiensi $E_p=S_p/p$, dan fraksi serial $f$ diestimasi dari hukum Amdahl.

In [ ]:
max_thr = numba.config.NUMBA_NUM_THREADS
threads = sorted({1, max_thr} | {t for t in (2, 4, 8, 16, 32) if t <= max_thr})
times = {}
for k in threads:
    numba.set_num_threads(k)
    times[k], _ = fs.benchmark_kernel(768, 30, "numba_parallel")
numba.set_num_threads(max_thr)

sc = pd.DataFrame({"thread": threads, "waktu (s)": [times[k] for k in threads]})
sc["speedup"] = times[1] / sc["waktu (s)"]
sc["efisiensi"] = sc["speedup"] / sc["thread"]
display(sc.round(3))

if max_thr > 1:
    S = sc["speedup"].iloc[-1]; p = max_thr
    f_serial = max(0.0, (1 / S - 1 / p) / (1 - 1 / p))
    print(f"Estimasi fraksi serial (Amdahl) pada {p} thread: f ≈ {f_serial:.3f}  ->  batas speedup teoretis ≈ {1/max(f_serial,1e-9):.1f}×")
else:
    print("Hanya 1 thread tersedia pada runtime ini; analisis skalabilitas thread tidak dapat dilakukan.")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(threads, sc["speedup"], "o-", lw=2, label="Terukur")
ax.plot(threads, threads, "k--", label="Ideal (linier)")
ax.set_xlabel("Jumlah thread"); ax.set_ylabel("Speedup"); ax.legend(); ax.set_title("Strong scaling kernel multi-thread")
ax.grid(alpha=0.3); plt.tight_layout(); savefig(fig, "06_strong_scaling"); plt.show()

## 6. Ensemble Paralel dan Peta Probabilitas
Prakiraan hujan dan debit sungai tidak pernah pasti. Maka dibangkitkan **8 skenario** dengan *Latin Hypercube Sampling* pada tiga parameter tak pasti: curah hujan total (90–250 mm), debit puncak sungai (250–900 m³/s), dan waktu puncak hujan (jam ke-1,5 s.d. 3,5). Setiap skenario adalah simulasi lengkap 6 jam pada grid 160 × 160 (resolusi 37,5 m).

Skenario saling bebas (*embarrassingly parallel*), sehingga dapat dibagi ke beberapa proses. Pada klaster nyata, proses-proses ini setara dengan *task* pada Slurm; di Colab dipakai multi-proses pada core CPU yang tersedia.

In [ ]:
N_ENS, N_SCEN = 160, 8
scen = fs.sample_scenarios(N_SCEN)
display(pd.DataFrame(scen).rename(columns={"id": "Skenario", "P": "Hujan total (mm)", "Q": "Debit puncak (m³/s)", "tp": "Puncak hujan (jam)"}).round(2))

# (a) Eksekusi serial: 1 proses
res_ser, t_ser = fs.run_ensemble(scen, n=N_ENS, workers=1, backend="numba_serial")
print(f"Serial    (1 proses)        : {t_ser:7.1f} detik")

# (b) Eksekusi paralel: beberapa proses (sudah termasuk biaya start proses + kompilasi JIT tiap proses)
N_WORKERS = max(2, N_CORES)
res_par, t_par = fs.run_ensemble(scen, n=N_ENS, workers=N_WORKERS, backend="numba_serial")
print(f"Paralel   ({N_WORKERS} proses)        : {t_par:7.1f} detik")

speedup = t_ser / t_par
print(f"Speedup = {speedup:.2f}×   |   efisiensi = {speedup / min(N_WORKERS, N_CORES):.0%} (terhadap {min(N_WORKERS, N_CORES)} core fisik/virtual)")

# (c) Validasi: hasil paralel harus identik dengan serial
diff = max(np.abs(a["hmax"] - b["hmax"]).max() for a, b in zip(res_ser, res_par))
print(f"Selisih maksimum hasil serial vs paralel: {diff:.1e} m  ->", "IDENTIK" if diff < 1e-6 else "BERBEDA")
print(f"Galat neraca massa maksimum seluruh skenario: {max(r['mass_err'] for r in res_par):.2e}")

fig, ax = plt.subplots(figsize=(5.5, 3.8))
b_ = ax.bar(["Serial\n(1 proses)", f"Paralel\n({N_WORKERS} proses)"], [t_ser, t_par], color=["#9e9e9e", "#59a14f"])
for r_, v in zip(b_, [t_ser, t_par]): ax.text(r_.get_x() + r_.get_width() / 2, v, f"{v:.1f} s", ha="center", va="bottom")
ax.set_ylabel("Waktu total ensemble (detik)"); ax.set_title(f"Ensemble {N_SCEN} skenario — speedup {speedup:.2f}×")
plt.tight_layout(); savefig(fig, "07_ensemble_speedup"); plt.show()

In [ ]:
dom_e = fs.make_domain(N_ENS)
H = np.stack([r["hmax"] for r in res_par])                       # (skenario, n, n)
TOA = np.stack([r["toa"] for r in res_par])
mask_e = np.where(dom_e["chan"], np.nan, 1.0)

P05 = (H > 0.5).mean(0) * mask_e
P10 = (H > 1.0).mean(0) * mask_e
Q90 = np.percentile(H, 90, axis=0) * mask_e
Le = dom_e["L"] / 1000.0; ext_e = [0, Le, Le, 0]

fig, ax = plt.subplots(1, 3, figsize=(17, 5.4))
for a, M, ttl, cm, vmax in [(ax[0], P05, "(a) Peluang kedalaman maks. > 0,5 m", "YlOrRd", 1),
                            (ax[1], P10, "(b) Peluang kedalaman maks. > 1,0 m", "YlOrRd", 1),
                            (ax[2], Q90, "(c) Persentil ke-90 kedalaman maks. (m)", "Blues", 2.5)]:
    a.imshow(dom_e["z"], cmap="Greys", extent=ext_e, alpha=0.35, vmin=-3, vmax=14)
    im = a.imshow(np.ma.masked_invalid(np.ma.masked_equal(M, 0)) if vmax == 1 else np.ma.masked_invalid(M), cmap=cm, vmin=0, vmax=vmax, extent=ext_e)
    a.set_title(ttl); a.set_xlabel("x (km)"); a.set_ylabel("y (km)"); plt.colorbar(im, ax=a, shrink=0.8)
fig.suptitle(f"Peta probabilistik dari ensemble {N_SCEN} skenario", fontsize=13)
plt.tight_layout(); savefig(fig, "08_peta_probabilitas"); plt.show()

flooded_share = [((np.nan_to_num(h_) > 0.5) & ~dom_e["chan"]).mean() * 100 for h_ in H]
print("Persentase luas daratan tergenang >0,5 m per skenario:", np.round(flooded_share, 1))

## 7. Buletin Peringatan Dini (peran Integrator)
Hasil ensemble diterjemahkan menjadi informasi yang dapat ditindaklanjuti pengguna awam. Untuk tiap zona dihitung:

- **Peluang terdampak** = proporsi skenario di mana ≥ 10% penduduk zona berada pada area dengan kedalaman maksimum > 0,5 m.
- **Penduduk terpapar rata-rata** pada seluruh skenario.
- **Perkiraan waktu tiba banjir** = median (lintas skenario) dari persentil ke-10 waktu tiba genangan > 0,3 m pada sel berpenduduk di zona tersebut.

Tingkat peringatan: **AWAS** (peluang ≥ 50%), **SIAGA** (≥ 25%), **WASPADA** (≥ 10%), **NORMAL** (di bawahnya).

In [ ]:
zid = dom_e["zone"].ravel()
pop_ze = np.bincount(zid, weights=dom_e["pop"].ravel(), minlength=16)
expo = np.array([np.bincount(zid, weights=(dom_e["pop"] * ((h_ > 0.5) & ~dom_e["chan"])).ravel(), minlength=16) for h_ in H])  # (S,16)
frac = expo / np.maximum(pop_ze, 1e-9)
p_impact = (frac >= 0.10).mean(0)

lead = np.full(16, np.nan)
populated = (dom_e["pop"] > 0.05) & ~dom_e["chan"]
for zi in range(16):
    vals = []
    for s in range(len(TOA)):
        m = (dom_e["zone"] == zi) & populated & ~np.isnan(TOA[s])
        if m.sum() >= 5:
            vals.append(np.percentile(TOA[s][m], 10) / 60.0)
    if vals: lead[zi] = np.median(vals)

def level(p):
    return "AWAS" if p >= 0.5 else "SIAGA" if p >= 0.25 else "WASPADA" if p >= 0.10 else "NORMAL"

bul = pd.DataFrame({"Zona": dom_e["zone_names"], "Penduduk": pop_ze.round(0).astype(int),
                    "Peluang terdampak (%)": (100 * p_impact).round(0).astype(int),
                    "Terpapar rata-rata (jiwa)": expo.mean(0).round(0).astype(int),
                    "Terpapar maks. (jiwa)": expo.max(0).round(0).astype(int),
                    "Tiba banjir (menit)": np.round(lead, 0),
                    "Level": [level(p) for p in p_impact]})
bul["Skor prioritas"] = (bul["Peluang terdampak (%)"] / 100 * bul["Terpapar rata-rata (jiwa)"]).round(0)
bul = bul.sort_values(["Skor prioritas", "Terpapar rata-rata (jiwa)"], ascending=False).reset_index(drop=True)
bul.index = bul.index + 1
bul.to_csv("hasil/buletin_peringatan_dini.csv", index_label="Prioritas")
display(bul)

In [ ]:
lvl_idx = {"NORMAL": 0, "WASPADA": 1, "SIAGA": 2, "AWAS": 3}
lvl_col = ["#c8e6c9", "#fff59d", "#ffb74d", "#ef5350"]
zl = np.array([lvl_idx[level(p)] for p in p_impact])
fig, ax = plt.subplots(1, 2, figsize=(13, 5.4))
ax[0].imshow(dom_e["z"], cmap="Greys", extent=ext_e, alpha=0.25, vmin=-3, vmax=14)
ax[0].imshow(zl[dom_e["zone"]], cmap=ListedColormap(lvl_col), vmin=-0.5, vmax=3.5, extent=ext_e, alpha=0.75, interpolation="nearest")
for r_ in range(4):
    for c_ in range(4):
        k = r_ * 4 + c_
        ax[0].text((c_ + 0.5) * Le / 4, (r_ + 0.5) * Le / 4, f"{dom_e['zone_names'][k]}\n{100*p_impact[k]:.0f}%", ha="center", va="center", fontsize=11, fontweight="bold")
ax[0].set_title("(a) Tingkat peringatan per zona (angka = peluang terdampak)")
ax[0].legend(handles=[Patch(color=c, label=n) for c, n in zip(lvl_col, lvl_idx)], loc="lower left", fontsize=8)
ax[0].set_xlabel("x (km)"); ax[0].set_ylabel("y (km)")

top = bul.sort_values("Terpapar rata-rata (jiwa)", ascending=True).tail(8)
cols = [lvl_col[lvl_idx[l]] for l in top["Level"]]
ax[1].barh(top["Zona"], top["Terpapar rata-rata (jiwa)"], color=cols, edgecolor="k")
ax[1].errorbar(top["Terpapar rata-rata (jiwa)"], top["Zona"], xerr=[np.zeros(len(top)), top["Terpapar maks. (jiwa)"] - top["Terpapar rata-rata (jiwa)"]], fmt="none", ecolor="k", capsize=3)
ax[1].set_xlabel("Penduduk terpapar (jiwa); garis = nilai maksimum ensemble"); ax[1].set_title("(b) Zona dengan paparan penduduk terbesar")
plt.tight_layout(); savefig(fig, "09_siaga_zona"); plt.show()

In [ ]:
n_awas = [z for z, p in zip(dom_e["zone_names"], p_impact) if level(p) == "AWAS"]
n_siaga = [z for z, p in zip(dom_e["zone_names"], p_impact) if level(p) == "SIAGA"]
n_wasp = [z for z, p in zip(dom_e["zone_names"], p_impact) if level(p) == "WASPADA"]
tot_mean, tot_max = expo.sum(1).mean(), expo.sum(1).max()
first = bul["Tiba banjir (menit)"].dropna()

buletin_teks = f'''
================ BULETIN PERINGATAN DINI BANJIR (SIMULASI) ================
Dasar prakiraan : ensemble {N_SCEN} skenario, hujan {min(s["P"] for s in scen):.0f}-{max(s["P"] for s in scen):.0f} mm/6 jam,
                  debit puncak sungai {min(s["Q"] for s in scen):.0f}-{max(s["Q"] for s in scen):.0f} m3/s
Status AWAS     : {", ".join(n_awas) if n_awas else "-"}
Status SIAGA    : {", ".join(n_siaga) if n_siaga else "-"}
Status WASPADA  : {", ".join(n_wasp) if n_wasp else "-"}
Penduduk terpapar (>0,5 m): rata-rata {tot_mean:,.0f} jiwa, skenario terburuk {tot_max:,.0f} jiwa
Genangan pertama diperkirakan tiba {first.min():.0f} menit setelah hujan mulai (zona {bul.loc[bul["Tiba banjir (menit)"].idxmin(), "Zona"]}).
Prioritas evakuasi : {", ".join(bul["Zona"].head(3))}
Catatan         : data bersifat sintetis; ini demonstrasi metodologi, bukan prakiraan operasional.
==========================================================================
'''
print(buletin_teks)
open("hasil/buletin_peringatan_dini.txt", "w", encoding="utf-8").write(buletin_teks)

### 7.1 Sensitivitas terhadap resolusi grid
Hasil model bergantung pada resolusi. Skenario Dasar dijalankan ulang pada empat resolusi untuk menunjukkan **harga komputasi** dan **perubahan hasil** saat grid diperhalus.

In [ ]:
rows = []
for n_ in (80, 120, 160):
    d_ = fs.make_domain(n_)
    r_ = fs.simulate(d_, fc, backend="numba_parallel")
    rows.append((n_, d_, r_))
rows.append((N, dom0, res))                                 # 240 sudah dijalankan sebelumnya
sens = []
for n_, d_, r_ in rows:
    m_ = (r_["hmax"] > 0.5) & ~d_["chan"]
    sens.append(dict(Grid=f"{n_}×{n_}", **{"Resolusi (m)": d_["dx"], "Langkah waktu": r_["steps"], "Waktu komputasi (s)": r_["runtime"],
                     "Luas tergenang >0,5 m (km²)": m_.sum() * (d_["dx"] / 1000) ** 2,
                     "Penduduk terpapar (jiwa)": d_["pop"][m_].sum(), "Kedalaman maks. (m)": np.nanmax(np.where(d_["chan"], np.nan, r_["hmax"]))}))
sens = pd.DataFrame(sens)
display(sens.round(2))
ref = sens.iloc[-1]
print("Selisih terhadap grid terhalus (240×240):")
for _, r_ in sens.iloc[:-1].iterrows():
    print(f"  {r_['Grid']:>8s}: luas {100*(r_['Luas tergenang >0,5 m (km²)']/ref['Luas tergenang >0,5 m (km²)']-1):+6.1f}% | penduduk terpapar {100*(r_['Penduduk terpapar (jiwa)']/ref['Penduduk terpapar (jiwa)']-1):+6.1f}% | waktu {r_['Waktu komputasi (s)']/ref['Waktu komputasi (s)']:.2f}× dari 240×240")

## 8. Skrip Klaster: Menjalankan di Superkomputer (Slurm)
Untuk skala yang melampaui Colab (ratusan skenario, grid lebih halus), kode yang sama dijalankan di klaster HPC. Berikut dua berkas: **`run_ensemble.py`** (antarmuka *command line* yang memakai modul `floodsim`) dan **`submit_ensemble.slurm`** (templat pekerjaan Slurm; nama partisi dan modul perlu disesuaikan dengan klaster yang dipakai).

In [ ]:
%%writefile run_ensemble.py
"""run_ensemble.py - Menjalankan ensemble skenario banjir dari command line (laptop, server, atau klaster Slurm)."""
import argparse
import os
import numpy as np
import floodsim as fs


def main():
    ap = argparse.ArgumentParser(description="Ensemble simulasi banjir 2D")
    ap.add_argument("--n", type=int, default=160, help="jumlah sel per sisi grid")
    ap.add_argument("--n-scen", type=int, default=8, help="jumlah skenario ensemble")
    ap.add_argument("--workers", type=int, default=2, help="jumlah proses paralel")
    ap.add_argument("--backend", default="numba_serial", choices=["numpy", "numba_serial"],
                    help="kernel yang dipakai tiap proses")
    ap.add_argument("--duration-h", type=float, default=6.0)
    ap.add_argument("--seed", type=int, default=2024)
    ap.add_argument("--out", default="hasil_cli")
    a = ap.parse_args()

    os.makedirs(a.out, exist_ok=True)
    scen = fs.sample_scenarios(a.n_scen, seed=a.seed)
    res, elapsed = fs.run_ensemble(scen, n=a.n, duration_h=a.duration_h,
                                   workers=a.workers, backend=a.backend)
    np.savez_compressed(
        os.path.join(a.out, "ensemble.npz"),
        hmax=np.stack([r["hmax"] for r in res]),
        toa=np.stack([r["toa"] for r in res]),
        P=[r["P"] for r in res], Q=[r["Q"] for r in res], tp=[r["tp"] for r in res])
    print(f"Selesai: {len(res)} skenario, grid {a.n}x{a.n}, {a.workers} proses, "
          f"waktu total {elapsed:.1f} s")
    print(f"Galat neraca massa maksimum: {max(r['mass_err'] for r in res):.2e}")
    print(f"Hasil disimpan di: {os.path.join(a.out, 'ensemble.npz')}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile submit_ensemble.slurm
#!/bin/bash
#SBATCH --job-name=flood-ensemble
#SBATCH --partition=compute            # sesuaikan dengan nama partisi di klaster yang dipakai
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64             # jumlah core CPU = jumlah proses paralel
#SBATCH --time=02:00:00
#SBATCH --output=flood_%j.out

# Lingkungan Python (nama modul bergantung pada klaster)
module load python
source ~/venv-flood/bin/activate

# Ensemble 256 skenario pada grid 480 x 480 (resolusi 12,5 m), satu skenario per core
srun python run_ensemble.py --n 480 --n-scen 256 --workers ${SLURM_CPUS_PER_TASK} \
     --out hasil_${SLURM_JOB_ID}

In [ ]:
# Uji asap (smoke test): jalankan CLI yang sama dengan yang akan dipakai Slurm, pada ukuran kecil
!python run_ensemble.py --n 80 --n-scen 2 --workers 2 --out hasil_cli
d = np.load("hasil_cli/ensemble.npz")
print("Isi berkas keluaran:", {k: d[k].shape for k in d.files})

### 8.1 Laporan HTML untuk pemangku kepentingan
Sebagai antarmuka akhir bagi pengguna awam (BPBD, perangkat kelurahan, warga), seluruh hasil dirangkum menjadi **satu berkas HTML mandiri** (gambar tertanam) yang dapat dibuka di peramban atau gawai tanpa instalasi apa pun.

In [ ]:
import base64, datetime
def img_tag(name, cap):
    b64 = base64.b64encode(open(f"hasil/{name}.png", "rb").read()).decode()
    return f'<figure><img src="data:image/png;base64,{b64}" style="max-width:100%"><figcaption>{cap}</figcaption></figure>'

col = {"AWAS": "#ef5350", "SIAGA": "#ffb74d", "WASPADA": "#fff59d", "NORMAL": "#c8e6c9"}
rows_html = "".join(
    f"<tr><td>{i}</td><td><b>{r['Zona']}</b></td><td>{r['Penduduk']:,}</td><td>{r['Peluang terdampak (%)']}%</td>"
    f"<td>{r['Terpapar rata-rata (jiwa)']:,}</td><td>{'' if np.isnan(r['Tiba banjir (menit)']) else int(r['Tiba banjir (menit)'])}</td>"
    f"<td style='background:{col[r['Level']]}'><b>{r['Level']}</b></td></tr>" for i, r in bul.iterrows())

html = f'''<!DOCTYPE html><html lang="id"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Laporan Peringatan Dini Banjir (Simulasi HPC)</title>
<style>body{{font-family:Arial,sans-serif;max-width:1000px;margin:auto;padding:16px;color:#222}}
table{{border-collapse:collapse;width:100%;font-size:14px}}td,th{{border:1px solid #ccc;padding:6px;text-align:center}}th{{background:#37474f;color:#fff}}
pre{{background:#f4f4f4;padding:12px;overflow-x:auto}}figcaption{{font-size:13px;color:#555;margin-bottom:18px}}</style></head><body>
<h1>Laporan Peringatan Dini Banjir Dataran Rendah</h1>
<p>Dihasilkan {datetime.datetime.now():%d-%m-%Y %H:%M} oleh simulasi 2D berbasis HPC (ensemble {N_SCEN} skenario). Data sintetis; demonstrasi metodologi.</p>
<h2>Buletin</h2><pre>{buletin_teks}</pre>
<h2>Tabel prioritas zona</h2>
<table><tr><th>Prioritas</th><th>Zona</th><th>Penduduk</th><th>Peluang terdampak</th><th>Terpapar rata-rata (jiwa)</th><th>Tiba banjir (menit)</th><th>Level</th></tr>{rows_html}</table>
<h2>Peta</h2>
{img_tag("09_siaga_zona", "Tingkat peringatan per zona dan paparan penduduk")}
{img_tag("08_peta_probabilitas", "Peta probabilistik dari ensemble")}
{img_tag("04_peta_bahaya_dasar", "Peta bahaya Skenario Dasar")}
</body></html>'''
open("hasil/laporan_peringatan_dini.html", "w", encoding="utf-8").write(html)
print("Laporan HTML dibuat:", "hasil/laporan_peringatan_dini.html", f"({os.path.getsize('hasil/laporan_peringatan_dini.html')/1e6:.1f} MB)")
display(HTML("<h4>Pratinjau tabel prioritas</h4><table style='font-size:13px'>" + rows_html.replace("<td", "<td style='border:1px solid #ccc;padding:3px'") + "</table>"))

# Paket hasil untuk diunduh (di Colab terunduh otomatis; di lingkungan lain tersimpan sebagai berkas zip)
shutil.make_archive("hasil_studi_kasus", "zip", "hasil")
try:
    from google.colab import files
    files.download("hasil_studi_kasus.zip")
except ImportError:
    print("Paket hasil:", "hasil_studi_kasus.zip")

## 9. Pembahasan, Keterbatasan, dan Kesimpulan

In [ ]:
best = piv.iloc[-1].drop("numpy")
best_name = best.idxmax()
print("RINGKASAN HASIL (dihitung otomatis dari eksekusi di atas)")
print("-" * 70)
print(f"1. Verifikasi model: neraca massa Skenario Dasar {res['mass_err']:.1e}; uji danau diam {q_resid:.1e} m²/s; uji hujan cekungan datar {rel:.2%}.")
print(f"2. Skenario Dasar menggenangi {int(exp_z.sum()):,} jiwa ({100*exp_z.sum()/pop_z.sum():.1f}% penduduk) pada kedalaman > 0,5 m.")
print(f"3. Kernel tercepat pada grid {piv.index.max()}²: {lab[best_name]} = {best.max():.0f} Mcell/s, "
      f"yaitu {best.max()/piv.iloc[-1]['numpy']:.1f}× lebih cepat dari NumPy 1 core.")
print(f"4. Ensemble {N_SCEN} skenario: {t_ser:.0f} s (serial) → {t_par:.0f} s ({N_WORKERS} proses), speedup {speedup:.2f}×, hasil identik.")
print(f"5. Simulasi 6 jam waktu nyata selesai dalam {res['runtime']:.0f} s ({6*3600/res['runtime']:.0f}× lebih cepat dari waktu nyata) — cukup cepat untuk peringatan dini.")
print(f"6. Ketidakpastian hujan/debit memengaruhi paparan: dari {expo.sum(1).min():,.0f} hingga {expo.sum(1).max():,.0f} jiwa antar skenario.")
print(f"7. Zona berstatus AWAS/SIAGA: {', '.join(n_awas + n_siaga) if (n_awas or n_siaga) else 'tidak ada'}.")

### 9.1 Pembahasan
- **Nilai tambah HPC.** Satu simulasi sudah jauh lebih cepat daripada waktu nyata, tetapi nilai terbesar muncul pada **ensemble**: peluang genangan per zona hanya dapat dihitung bila puluhan skenario dijalankan dalam waktu yang sama dengan satu simulasi. Bagian 6 menunjukkan bahwa pembagian skenario ke beberapa proses menghasilkan keluaran yang **identik** dengan eksekusi serial, sehingga paralelisasi tidak mengorbankan akurasi.
- **Tiga tingkat paralelisme saling melengkapi.** Paralelisme data (thread/GPU) mempercepat satu simulasi; paralelisme tugas mempercepat ensemble. Pada klaster, keduanya dapat digabung: satu node GPU per kelompok skenario, atau satu core per skenario.
- **Peran skalabilitas.** Pada ukuran grid kecil, kinerja GPU dan multi-thread tidak optimal karena beban kerja tidak cukup untuk menutupi biaya peluncuran kernel dan sinkronisasi. Keuntungan membesar pada grid yang lebih besar. Karena beban kerja tumbuh ∝ $n^3$, resolusi tinggi baru praktis bila ada akselerasi.
- **Sensitivitas resolusi.** Sungai selebar 120 m tidak terwakili baik pada grid kasar, sehingga hasil bergeser terhadap grid terhalus (Bagian 7.1). Inilah alasan resolusi tinggi diperlukan, dan alasan ia membutuhkan HPC.
- **Dari angka menjadi keputusan.** Buletin mengubah keluaran numerik menjadi level peringatan, prioritas evakuasi, dan estimasi waktu tiba, yaitu bentuk informasi yang dapat dipakai pengguna non-teknis. Inilah inti peran *Integrator* pada kuliah praktisi.

### 9.2 Keterbatasan
1. **Data sintetis**: domain, penduduk, dan zona fiktif; hasil tidak dapat dipakai sebagai prakiraan wilayah nyata. Untuk wilayah nyata, DEM resolusi tinggi, data tutupan lahan, kependudukan, dan debit sungai terukur harus mengganti array sintetis, lalu model dikalibrasi dan divalidasi terhadap kejadian banjir historis.
2. **Fisika disederhanakan**: hujan seragam, koefisien limpasan konstan, tanpa jaringan drainase, infiltrasi, bangunan, atau pasang surut; gesekan memakai Manning dengan suku adveksi diabaikan.
3. **Ensemble kecil** (8 skenario) hanya untuk demonstrasi; peluang yang dilaporkan berkelipatan 12,5%. Penerapan operasional membutuhkan ratusan skenario, yang justru memanfaatkan skalabilitas yang ditunjukkan di sini.
4. **Kinerja bergantung perangkat keras**: Colab gratis hanya menyediakan sedikit core CPU, sehingga *speedup* multi-proses/thread terbatas pada jumlah core tersebut. Skrip Slurm di Bagian 8 disediakan untuk skala yang lebih besar, tetapi tidak dijalankan di Colab.
5. Kernel GPU memakai presisi ganda (`float64`); pada GPU kelas konsumen/Colab, presisi tunggal dapat jauh lebih cepat dan layak dipertimbangkan bila akurasi mengizinkan.

### 9.3 Kesimpulan
Notebook ini menunjukkan alur kerja lengkap *Integrator*: data lapangan dikonversi, diverifikasi, disimulasikan secara masif dengan kernel paralel, dianalisis secara probabilistik, lalu disajikan sebagai buletin dan laporan HTML. Model lolos verifikasi (kekekalan massa dan keseimbangan), kernel multi-thread/GPU memberi percepatan terukur dibandingkan NumPy, dan ensemble paralel menghasilkan keluaran identik dengan eksekusi serial dalam waktu lebih singkat. Dengan demikian pernyataan inti kuliah terbukti pada skala kecil: **infrastruktur komputasi kinerja tinggi dapat menjadi jembatan bagi persoalan lokal** seperti peringatan dini banjir, bahkan dari mahasiswa di perguruan tinggi daerah menggunakan sumber daya gratis dan perangkat lunak terbuka.

### 9.4 Daftar Pustaka
- Diaz Carballo, W. *Bridging Global Infrastructure to Address Local Challenge with HPC*. Kuliah Praktisi Informatika, Prodi S1 Teknik Informatika, Universitas Bengkulu (rekaman: https://www.youtube.com/live/qeWpf9uXBqE).
- Bates, P. D., Horritt, M. S., & Fewtrell, T. J. (2010). A simple inertial formulation of the shallow water equations for efficient two-dimensional flood inundation modelling. *Journal of Hydrology*, 387(1–2), 33–45.
- Lam, S. K., Pitrou, A., & Seibert, S. (2015). Numba: A LLVM-based Python JIT compiler. *Proceedings of the Second Workshop on the LLVM Compiler Infrastructure in HPC*.
- Amdahl, G. M. (1967). Validity of the single processor approach to achieving large scale computing capabilities. *AFIPS Conference Proceedings*, 30, 483–485.
- Courant, R., Friedrichs, K., & Lewy, H. (1928). Über die partiellen Differenzengleichungen der mathematischen Physik. *Mathematische Annalen*, 100, 32–74.
- Okuta, R., dkk. (2017). CuPy: A NumPy-compatible library for NVIDIA GPU calculations. *Proceedings of Workshop on Machine Learning Systems (LearningSys) at NIPS 2017*.